# 05 — Business Insights

This notebook consolidates funnel and conversion outputs into stakeholder-ready findings and prioritized opportunities.

## Coverage
- KPI summary
- Funnel drop-off and conversion opportunities
- Segment ranking
- Scenario analysis
- Executive visualizations
- Final metrics and recommendation exports

> Retailrocket provides no real order value or attribution metadata. `revenue` is a purchase indicator normalized to `1.00` and must not be interpreted as currency.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "data" / "ecommerce_funnel_events.csv"
REPORTS_PATH = ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True, errors="coerce")

stages = ["product_view", "cart", "purchase"]
counts = df.groupby("stage")["user_id"].nunique().reindex(stages)

funnel = pd.DataFrame({
    "stage": stages,
    "users": [int(counts[stage]) for stage in stages],
})
funnel["step_conversion"] = funnel["users"].div(funnel["users"].shift(1)).fillna(1.0)
funnel["overall_conversion"] = funnel["users"] / funnel["users"].iloc[0]
funnel["drop_off_users"] = funnel["users"].shift(1) - funnel["users"]
funnel["drop_off_users"] = funnel["drop_off_users"].fillna(0).astype(int)

display(funnel)

purchase_count = int(counts["purchase"])
average_indicator = df.loc[df["stage"] == "purchase", "revenue"].astype(float).mean() if purchase_count else 0.0

opportunities = []
for i in range(len(stages) - 1):
    source = set(df.loc[df["stage"] == stages[i], "user_id"])
    target = set(df.loc[df["stage"] == stages[i + 1], "user_id"])
    abandoned = len(source - target)
    opportunities.append({
        "transition": f"{stages[i]} → {stages[i + 1]}",
        "abandoned_users": abandoned,
        "normalized_indicator_value_per_user": round(float(average_indicator), 2),
        "estimated_recovery_indicator": round(abandoned * float(average_indicator), 2),
    })

opportunity_table = pd.DataFrame(opportunities).sort_values("abandoned_users", ascending=False)
recommendations = pd.DataFrame([
    {"priority": 1, "area": "Largest funnel drop-off", "recommendation": "Diagnose friction, relevance, and tracking at the largest observed transition; validate with a randomized experiment.", "primary_metric": "step conversion rate"},
    {"priority": 2, "area": "Cart completion", "recommendation": "Test cart reminders or a simplified return-to-cart experience for visitors who reach the cart stage.", "primary_metric": "cart-to-purchase conversion"},
    {"priority": 3, "area": "Measurement", "recommendation": "Replace normalized revenue and unknown attribution fields with real order value, acquisition channel, device, and campaign data.", "primary_metric": "data completeness and incrementality"},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(funnel["stage"], funnel["users"], marker="o", linewidth=2, color="#4C78A8")
axes[0].set_title("Funnel Volume")
axes[0].set_ylabel("Unique users")
axes[0].tick_params(axis="x", rotation=30)

opportunity_table.plot(kind="bar", x="transition", y="abandoned_users", ax=axes[1], legend=False, color="#F28E2B")
axes[1].set_title("Potential Recovery Opportunity")
axes[1].set_ylabel("Abandoned users")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(REPORTS_PATH / "executive_funnel_dashboard.png", dpi=150)
plt.show()

kpis = pd.DataFrame([
    {"kpi": "product_view_users", "value": int(counts["product_view"])},
    {"kpi": "cart_users", "value": int(counts["cart"])},
    {"kpi": "purchase_users", "value": purchase_count},
    {"kpi": "product_view_to_purchase_rate", "value": purchase_count / int(counts["product_view"]) if counts["product_view"] else 0.0},
])

kpis.to_csv(REPORTS_PATH / "business_kpis.csv", index=False)
funnel.to_csv(REPORTS_PATH / "business_funnel_metrics.csv", index=False)
opportunity_table.to_csv(REPORTS_PATH / "business_opportunity_metrics.csv", index=False)
recommendations.to_csv(REPORTS_PATH / "business_recommendations.csv", index=False)

display(kpis)
display(opportunity_table)
display(recommendations)
print("Figures and tables exported to:", REPORTS_PATH)
